<a href="https://colab.research.google.com/github/MuayThaiLegz/PracticeCrazy/blob/main/GatherResolutionsXSessions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyreadr PyPDF2 langchain_community openai huggingface_hub pdfplumber langchain_openai PyMuPDF

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 524.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.0/417.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0

In [15]:
import os
import requests
import random
import time
# import pdfplumber
import unicodedata
# import pyreadr
import re
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from io import BytesIO
import math
from concurrent.futures import ThreadPoolExecutor, as_completed

import asyncio
import aiohttp
import nest_asyncio
import requests
import pandas as pd
from bs4 import BeautifulSoup


from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from PyPDF2 import PdfReader

# Ensure the settings for pandas display are configured


In [74]:
save_dir_pdfs = '/content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs'
text_output_dir = '/content/drive/MyDrive/EcoLawCollection/DataCollection/un_extracted_texts'


df_download_links = pd.read_csv("resolution_download_links.csv")
resolution_info_df = pd.read_csv('df_resolutions.csv').drop(columns=['Unnamed: 0'])


In [69]:
import os
import math
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

class PDFDownloader:
    def __init__(self, df_download_links, save_dir_pdfs, batch_size=361, max_workers=10, failed_downloads_file='failed_downloads.txt', retry_failed_downloads_file='retry_failed_downloads.txt'):
        self.df_download_links = df_download_links
        self.save_dir_pdfs = save_dir_pdfs
        self.batch_size = batch_size
        self.max_workers = max_workers
        self.failed_downloads_file = failed_downloads_file
        self.retry_failed_downloads_file = retry_failed_downloads_file
        self.total_batches = math.ceil(len(df_download_links) / batch_size)

        # Ensure save directories exist
        if not os.path.exists(self.save_dir_pdfs):
            os.makedirs(self.save_dir_pdfs)

    def log_failed_download(self, url, status_code, retry=False):
        """Log failed download attempts to a file."""
        log_file = self.retry_failed_downloads_file if retry else self.failed_downloads_file
        with open(log_file, 'a') as file:
            file.write(f"Failed to download {url}, status code: {status_code}\n")

    def download_pdf(self, url, file_path):
        """Download a PDF file from the given URL and save it to the specified path if it doesn't already exist."""
        if os.path.exists(file_path):
            print(f"File already exists, skipping download: {file_path}")
            return

        try:
            response = requests.get(url, stream=True)
            if response.status_code == 200:
                with open(file_path, 'wb') as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)
                print(f"Successfully downloaded: {file_path}")
            else:
                print(f"Failed to download {url}, status code: {response.status_code}")
                self.log_failed_download(url, response.status_code)
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            self.log_failed_download(url, 'Exception')

    def process_batch(self, batch_number):
        """Process a batch of PDF downloads using parallel execution."""
        start_idx = batch_number * self.batch_size
        end_idx = min(start_idx + self.batch_size, len(self.df_download_links))
        batch_df = self.df_download_links.iloc[start_idx:end_idx]

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_url = {
                executor.submit(
                    self.download_pdf,
                    row['Download Link'],
                    os.path.join(self.save_dir_pdfs, f"{row['Resolution Number'].replace('/', '_').replace(' ', '_')}.pdf")
                ): row['Resolution Number']
                for _, row in batch_df.iterrows()
            }

            for future in as_completed(future_to_url):
                resolution_number = future_to_url[future]
                try:
                    future.result()
                except Exception as e:
                    print(f"Error occurred while processing {resolution_number}: {e}")
                    self.log_failed_download(resolution_number, 'Exception')

    def process_all_batches(self):
        """Process all batches of PDF downloads."""
        for batch_number in range(self.total_batches):
            print(f"Processing batch {batch_number + 1}/{self.total_batches}...")
            self.process_batch(batch_number)
            print(f"Completed batch {batch_number + 1}/{self.total_batches}")

    def retry_failed_downloads(self):
        """Retry downloading PDFs listed in the failed_downloads.txt."""
        if not os.path.exists(self.failed_downloads_file):
            print(f"No failed downloads to retry. File '{self.failed_downloads_file}' not found.")
            return

        with open(self.failed_downloads_file, 'r') as file:
            lines = file.readlines()

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_url = {}
            for line in lines:
                parts = line.strip().split(", status code:")
                if len(parts) > 1:
                    url = parts[0].replace("Failed to download ", "").strip()
                    resolution_number = url.split("DS=")[-1].replace('&Lang=E', '').replace('/', '_').replace(' ', '_')
                    pdf_file_path = os.path.join(self.save_dir_pdfs, f"{resolution_number}.pdf")

                    future = executor.submit(self.download_pdf, url, pdf_file_path)
                    future_to_url[future] = url

            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    future.result()
                except Exception as e:
                    print(f"Error occurred while processing {url}: {e}")
                    self.log_failed_download(url, 'Exception', retry=True)



Streaming output truncated to the last 5000 lines.
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_247.pdf
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_246.pdf
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_248.pdf
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_252.pdf
Failed to download https://daccess-ods.un.org/access.nsf/Get?OpenAgent&DS=A/RES/51/222%20A&Lang=E, status code: 404
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_249.pdf
File already exists, skipping download: /content/drive/MyDrive/EcoLawCollection/DataCollection/un_resolutions_pdfs/A_RES_52_250.pdf
File already exists, skipping download: /

In [70]:

class PDFTextExtractor:
    def __init__(self, save_dir_pdfs, text_output_dir, max_workers=10):
        """
        Initialize the PDFTextExtractor with the directories for PDFs and text output.

        Args:
            save_dir_pdfs (str): Directory where the PDF files are stored.
            text_output_dir (str): Directory where extracted text files will be saved.
            max_workers (int): Number of workers for concurrent processing.
        """
        self.save_dir_pdfs = save_dir_pdfs
        self.text_output_dir = text_output_dir
        self.max_workers = max_workers

        # Ensure the text output directory exists
        if not os.path.exists(self.text_output_dir):
            os.makedirs(self.text_output_dir)

    def extract_text_from_pdf(self, pdf_path, text_output_path):
        """
        Extract text from a PDF file and save it to a text file.

        Args:
            pdf_path (str): Path to the PDF file.
            text_output_path (str): Path to save the extracted text file.
        """
        try:
            reader = PdfReader(pdf_path)  # Open the PDF using PyPDF2
            text = ""
            for page in reader.pages:  # Iterate through each page in the PDF
                page_text = page.extract_text()
                if page_text:  # Check if there is text on the page
                    text += page_text + "\n"

            if text.strip():  # Only write to file if text is extracted
                with open(text_output_path, 'w', encoding='utf-8') as text_file:
                    text_file.write(text)
                print(f"Text successfully extracted from {pdf_path} to {text_output_path}")
            else:
                print(f"No text found in {pdf_path}")
        except Exception as e:
            print(f"Error extracting text from {pdf_path}: {e}")

    def process_pdf(self, pdf_file):
        """
        Process a single PDF file for text extraction.

        Args:
            pdf_file (str): Name of the PDF file to process.
        """
        pdf_path = os.path.join(self.save_dir_pdfs, pdf_file)
        text_file_name = os.path.splitext(pdf_file)[0] + '.txt'
        text_output_path = os.path.join(self.text_output_dir, text_file_name)

        # Extract text only if the text file does not already exist
        if not os.path.exists(text_output_path):
            self.extract_text_from_pdf(pdf_path, text_output_path)
        else:
            print(f"Text file already exists, skipping extraction for: {text_output_path}")

    def process_all_pdfs_concurrently(self):
        """Process all PDFs in the directory concurrently."""
        pdf_files = [file for file in os.listdir(self.save_dir_pdfs) if file.lower().endswith('.pdf')]

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_pdf = {executor.submit(self.process_pdf, pdf_file): pdf_file for pdf_file in pdf_files}

            for future in as_completed(future_to_pdf):
                pdf_file = future_to_pdf[future]
                try:
                    future.result()  # This will raise any exception that occurred during execution
                except Exception as e:
                    print(f"Error occurred while processing {pdf_file}: {e}")

In [72]:
pdf_downloader = PDFDownloader(df_download_links, save_dir_pdfs=save_dir_pdfs)
pdf_downloader.process_all_batches()  # Start the batch processing
pdf_downloader.retry_failed_downloads()  # Retry failed downloads
pdf_text_extractor = PDFTextExtractor(save_dir_pdfs=save_dir_pdfs, text_output_dir=text_output_dir, max_workers=10)
pdf_text_extractor.process_all_pdfs_concurrently()  # Start processing all PDFs concurrently

In [79]:


data = []

# Iterate through each file in the text output directory
for file_name in os.listdir(text_output_dir):
    if file_name.endswith('.txt'):
        resolution_number = os.path.splitext(file_name)[0]  # Extract resolution number from file name
        text_file_path = os.path.join(text_output_dir, file_name)

        # Read the content of the text file
        with open(text_file_path, 'r', encoding='utf-8') as file:
            text_content = file.read()

        # Append the resolution number and text content to the data list
        data.append({'Resolution Number': resolution_number, 'Text': text_content})

# Create a DataFrame from the data
df_resolutions_text = pd.DataFrame(data)

# Display the first few rows of the DataFrame
df_resolutions_text.head()

,Resolution Number,Text
0,A_RES_70_301,United Nations A/RES/ 70/301 \n General Ass...
1,A_RES_69_204,United Nations A/RES/69/204 \n General Assemb...
2,A_RES_69_207,United Nations A/RES/69/207 \n General Assemb...
3,A_RES_69_208,United Nations A/RES/69/208 \n General Assemb...
4,A_RES_69_210,United Nations A/RES/69/210 \n General Assemb...
